### VPC Flow Logs Ingestionator
Ingest VPC Flow logs directly from S3 into a dataframe. Part of the DUNE project (https://github.com/opendr-io/dune) for hunting threats that are resistant to conventional detection. Run this in the same region!  Populate the target S3 URI below. I'm assuming credentials are already populated for the AWS client because this is probably running in the same region as the data.

In [ ]:
import io, gzip, json, pandas as pd
import re, io, json, gzip
import boto3
import pandas as pd
import re, io, json, gzip, boto3
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [ ]:
# Find the right bucket / prefix without scrolling ~500 bucket names.
# find_buckets() filters bucket names client-side; ls() mimics
# `aws s3 ls s3://.../` ONE level at a time (Delimiter="/") so you can walk
# down to the flow-log path.
def find_buckets(*substrings, ignore_case=True):
    names = [b["Name"] for b in boto3.client("s3").list_buckets()["Buckets"]]
    if not substrings:
        return names
    def hit(n):
        h = n.lower() if ignore_case else n
        return any((s.lower() if ignore_case else s) in h for s in substrings)
    return [n for n in names if hit(n)]

def ls(s3_uri, max_keys=40):
    m = re.match(r"^s3://([^/]+)/?(.*)$", s3_uri)
    bucket, prefix = m.group(1), m.group(2)
    resp = boto3.client("s3").list_objects_v2(
        Bucket=bucket, Prefix=prefix, Delimiter="/", MaxKeys=max_keys
    )
    for cp in resp.get("CommonPrefixes", []):
        print("        PRE  " + cp["Prefix"])
    for obj in resp.get("Contents", []):
        if obj["Key"] == prefix:
            continue
        print(f"{obj['Size']:>12,}  {obj['Key']}")
    if resp.get("IsTruncated"):
        print("  ... (truncated -- raise max_keys)")

# Flow logs usually live under:
#   s3://<bucket>/[prefix/]AWSLogs/<account-id>/vpcflowlogs/<region>/<yyyy>/<mm>/<dd>/
print("candidate buckets:")
for b in find_buckets("flow", "vpc", "log"):
    print(" -", b)

# then drill down, e.g.:
# ls("s3://<bucket>/")
# ls("s3://<bucket>/AWSLogs/")


In [ ]:
# Populate a flow logs URI
S3_URI = "s3://.."


In [ ]:
# Interrogate the URI and calculate the data volume there
SUFFIX = ".log.gz"   # plain-text VPC Flow Logs; use ".parquet" if delivered as parquet

m = re.match(r"^s3://([^/]+)/(.*)$", S3_URI.rstrip("/"))
S3_BUCKET, S3_PREFIX = (m.group(1), m.group(2) + "/") if m else (S3_URI[5:], "")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

all_objs = []
other_keys = 0
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(SUFFIX):
            all_objs.append(obj)
        else:
            other_keys += 1

print(f"Found {len(all_objs)} '{SUFFIX}' files under {S3_URI}")
if other_keys:
    print(f"({other_keys} other objects under the prefix were ignored)")

total_bytes = sum(obj["Size"] for obj in all_objs)
total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print(f"Total size: {total_bytes:,} bytes")
print(f"\u2248 {total_mb:.2f} MB  ({total_gb:.3f} GB)")

In [ ]:
# Enumerate fields present in the VPC Flow Log files.
# Plain-text flow logs carry their schema as a header line (the first row of
# each .log.gz). Custom formats are possible, so sample several files, union
# their headers, and flag any file whose header layout differs.
def list_vpcflow_fields(s3_uri, access_key=None, secret_key=None, limit=10):
    m = re.match(r"^s3://([^/]+)/?(.*)$", s3_uri.rstrip("/"))
    if not m:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    bucket, prefix = m.group(1), (m.group(2) + "/") if m.group(2) else ""

    s3 = boto3.client(
        "s3",
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
    )

    paginator = s3.get_paginator("list_objects_v2")
    header_by_file = {}
    files_read = 0

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith(".log.gz"):
                continue

            body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
            with gzip.GzipFile(fileobj=io.BytesIO(body)) as gz:
                first_line = gz.readline().decode("utf-8").strip()
            if first_line:
                header_by_file[obj["Key"]] = first_line.split()

            files_read += 1
            if limit and files_read >= limit:
                break
        if limit and files_read >= limit:
            break

    if not header_by_file:
        print("No .log.gz files found under the prefix.")
        return []

    distinct = {tuple(h) for h in header_by_file.values()}
    if len(distinct) > 1:
        print(f"WARNING: {len(distinct)} distinct header layouts across {files_read} sampled files:")
        for h in distinct:
            print("   ", " ".join(h))
        print()

    field_names = set()
    for h in header_by_file.values():
        field_names.update(h)

    return sorted(field_names)

# --- Example ---
fields = list_vpcflow_fields(
    S3_URI,
    limit=5,   # read first 5 files to sample
)
print(f"Discovered {len(fields)} available fields:")
for f in fields:
    print("-", f)

In [ ]:
# Functions to read gzipped VPC Flow Logs from a bucket directly into a dataframe
def _parse_s3_uri(s3_uri: str):
    m = re.match(r"^s3://([^/]+)/?(.*)$", s3_uri.rstrip("/"))
    if not m:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    bucket, prefix = m.group(1), (m.group(2) + "/") if m.group(2) else ""
    return bucket, prefix

def _make_s3_client(access_key=None, secret_key=None, session_token=None, region_name=None):
    # Falls back to default credential chain if keys are None
    session = boto3.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        aws_session_token=session_token,
        region_name=region_name,
    )
    return session.client("s3")

# VPC Flow Log fields that are numeric when present (default + common custom formats)
_VPCFLOW_NUMERIC = [
    "version", "srcport", "dstport", "protocol", "packets", "bytes",
    "start", "end", "tcp-flags", "traffic-path",
]

def load_vpcflow_vflows_from_s3_uri(
    s3_uri: str,
    access_key=None,
    secret_key=None,
    session_token=None,
    region_name=None,
    suffix=".log.gz",
    limit=None,             # set an int to sample files first
    select_cols=None,       # list of columns to keep
    na_value="-",           # VPC Flow Logs use "-" for missing values
    verbose=True,           # print progress + a report of anything skipped/coerced
) -> pd.DataFrame:
    """
    Recursively loads plain-text (gzipped) VPC Flow Logs from an S3 URI into a
    DataFrame. Each .log.gz carries its own header row as its first line, so
    files written under different custom formats are unioned column-wise.

    Unreadable files are skipped and unparseable numeric/timestamp values are
    coerced to <NA>; both are reported at the end when verbose=True.
    """
    bucket, prefix = _parse_s3_uri(s3_uri)
    s3 = _make_s3_client(access_key, secret_key, session_token, region_name)

    paginator = s3.get_paginator("list_objects_v2")
    frames = []
    file_count = 0
    skipped = []            # (key, reason) for files that could not be parsed

    cur_dir = None
    dir_files = 0
    dir_rows = 0
    dir_skipped = 0

    def _flush_dir():
        if verbose and cur_dir is not None:
            extra = f", {dir_skipped} skipped" if dir_skipped else ""
            print(f"  finished {cur_dir}/  ->  {dir_files} file(s), {dir_rows:,} rows{extra}")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith(suffix):
                continue

            file_count += 1
            if limit and file_count > limit:
                break

            folder = key.rsplit("/", 1)[0] if "/" in key else ""
            if folder != cur_dir:
                _flush_dir()
                cur_dir, dir_files, dir_rows, dir_skipped = folder, 0, 0, 0

            try:
                body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
                with gzip.GzipFile(fileobj=io.BytesIO(body)) as gz:
                    frame = pd.read_csv(gz, sep=" ", dtype=str, na_values=[na_value])
            except (pd.errors.EmptyDataError, pd.errors.ParserError, OSError, gzip.BadGzipFile) as e:
                skipped.append((key, type(e).__name__))
                dir_skipped += 1
                continue

            dir_files += 1
            if not frame.empty:
                frames.append(frame)
                dir_rows += len(frame)
        if limit and file_count >= limit:
            break

    _flush_dir()

    if skipped and verbose:
        print(f"\nskipped {len(skipped)} unreadable file(s):")
        for k, reason in skipped[:10]:
            print(f"  - {k}  ({reason})")
        if len(skipped) > 10:
            print(f"  ... and {len(skipped) - 10} more")

    if not frames:
        if verbose:
            print("no records read")
        return pd.DataFrame()

    vflows = pd.concat(frames, ignore_index=True)

    # Coerce known-numeric columns; nullable Int64 so missing values survive.
    # Track values that were present but failed to parse (became <NA>).
    coerced = {}
    for col in _VPCFLOW_NUMERIC:
        if col in vflows.columns:
            had_value = vflows[col].notna()
            conv = pd.to_numeric(vflows[col], errors="coerce")
            lost = int((conv.isna() & had_value).sum())
            if lost:
                coerced[col] = lost
            vflows[col] = conv.astype("Int64")

    # start / end are Unix epoch seconds -> add UTC datetime companions
    for col in ("start", "end"):
        if col in vflows.columns:
            had_value = vflows[col].notna()
            conv = pd.to_datetime(vflows[col], unit="s", utc=True, errors="coerce")
            lost = int((conv.isna() & had_value).sum())
            if lost:
                coerced[f"{col}_time"] = lost
            vflows[f"{col}_time"] = conv

    if coerced and verbose:
        total = sum(coerced.values())
        detail = ", ".join(f"{k}={v}" for k, v in coerced.items())
        print(f"coerced {total:,} unparseable value(s) to <NA>: {detail}")

    # Optional column pruning
    if select_cols:
        keep = [c for c in select_cols if c in vflows.columns]
        if keep:
            vflows = vflows[keep]

    if verbose:
        print(f"done: {len(vflows):,} rows from {len(frames)} file(s)")

    return vflows


In [ ]:
# Read VPC Flow Logs from the target S3 URI and keep the listed fields.
# Reconcile this list with whatever `list_vpcflow_fields` printed above -- these
# are the AWS default-format fields plus the derived *_time columns added by the
# loader. Drop `select_cols` entirely to keep every column the headers expose.
vflows = load_vpcflow_vflows_from_s3_uri(
    S3_URI,
    region_name="us-east-1",  # optional
    select_cols=[
        "version",
        "account-id",
        "interface-id",
        "srcaddr",
        "dstaddr",
        "srcport",
        "dstport",
        "protocol",
        "packets",
        "bytes",
        "start",
        "end",
        "start_time",
        "end_time",
        "action",
        "log-status",
    ],
)

print(vflows.shape)
vflows.head()


In [ ]:
# Describe the dataframe that has been produced for inspection
# Top level fields should not be entirely NaN apart from error fields

def summarize_sparse_columns(vflows, threshold=0.95):
    total_rows = len(vflows)
    nan_counts = vflows.isna().sum()

    # Entirely NaN
    all_nan = nan_counts[nan_counts == total_rows].index.tolist()

    # Mostly NaN (but not entirely)
    mostly_nan = nan_counts[(nan_counts / total_rows >= threshold) & (nan_counts < total_rows)].index.tolist()

    print(f"Total rows: {total_rows:,}")
    print(f"\n🧱 Columns entirely NaN ({len(all_nan)}):")
    for c in all_nan:
        print("  -", c)

    print(f"\n🌫️ Columns ≥ {int(threshold*100)}% NaN ({len(mostly_nan)}):")
    for c in mostly_nan:
        pct = nan_counts[c] / total_rows * 100
        print(f"  - {c}: {pct:.1f}% NaN")

    return all_nan, mostly_nan

# --- Example usage ---
all_nan_cols, mostly_nan_cols = summarize_sparse_columns(vflows, threshold=0.95)

empty_cols = [col for col in vflows.columns if vflows[col].isna().all()]
print()
print(f"{len(empty_cols)} columns are entirely NaN:")
for c in empty_cols:
    print("-", c)
    
print()
vflows.info(verbose=True)


In [ ]:
# Cast fields and timestamps
vflows['bytes'] = pd.to_numeric(vflows['bytes'], errors='coerce')
vflows['packets'] = pd.to_numeric(vflows['packets'], errors='coerce')
vflows['start'] = pd.to_datetime(vflows['start'], unit='s')
vflows['end'] = pd.to_datetime(vflows['end'], unit='s')
columns = ["start", "end"] + [col for col in vflows.columns if col not in [ "start", "end"]]
vflows = vflows[columns]
#vflows.head()

In [ ]:
# Prepare counts for plotting

vflows['bytes'] = pd.to_numeric(vflows['bytes'], errors='coerce')
vflows['packets'] = pd.to_numeric(vflows['packets'], errors='coerce')
vflows = vflows.dropna(subset=['bytes', 'packets'])


dates = pd.to_datetime(vflows['start']).to_numpy()  
bytes_arr = vflows['bytes'].to_numpy()
packets_arr = vflows['packets'].to_numpy()

months = np.array([pd.Timestamp(d).strftime('%Y-%m') for d in dates])  
dates_exact = np.array([pd.Timestamp(d).strftime('%Y-%m-%d') for d in dates])  
unique_dates, inverse_indices = np.unique(dates.astype('datetime64[D]'), return_inverse=True)

flows_count = np.bincount(inverse_indices)  
bytes_sum = np.bincount(inverse_indices, weights=bytes_arr)  
packets_sum = np.bincount(inverse_indices, weights=packets_arr)  

unique_months = np.array([pd.Timestamp(d).strftime('%Y-%m') for d in unique_dates])
unique_dates_exact = np.array([pd.Timestamp(d).strftime('%Y-%m-%d') for d in unique_dates])

result = np.rec.fromarrays(
    [unique_dates_exact, unique_months, flows_count, bytes_sum, packets_sum],
    names=('date', 'month', 'Flows', 'Bytes', 'Packets')
)

data = pd.DataFrame(result)
print(data)



In [ ]:
# Plot flow count 30 day moving average with BBs

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')
data['Flows_30D_MA'] = data['Flows'].rolling(window=30).mean()
data['Flows_30D_STD'] = data['Flows'].rolling(window=30).std()
data['Upper_Band'] = data['Flows_30D_MA'] + (2 * data['Flows_30D_STD'])
data['Lower_Band'] = data['Flows_30D_MA'] - (2 * data['Flows_30D_STD'])

def comma_formatter(x, _):
    return f'{x:,.0f}'

plt.figure(figsize=(12, 6))
plt.plot(data['date'], data['Flows'], label='Daily Flows', color='red', alpha=0.6)
plt.plot(data['date'], data['Flows_30D_MA'], label='30-Day Moving Average', color='darkblue', linewidth=2)
plt.fill_between(data['date'], data['Upper_Band'], data['Lower_Band'], color='grey', alpha=0.3, label='Bollinger Bands')
plt.title('Flow Counts with 30-Day Moving Average and Bollinger Bands', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Flow Counts', fontsize=12)

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=10, frameon=False)
plt.gca().yaxis.set_major_formatter(FuncFormatter(comma_formatter))
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Plot byte count 30 day moving average with BBs

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')
data['Bytes_30D_MA'] = data['Bytes'].rolling(window=30).mean()
data['Bytes_30D_STD'] = data['Bytes'].rolling(window=30).std()
data['Bytes_Upper_Band'] = data['Bytes_30D_MA'] + (2 * data['Bytes_30D_STD'])
data['Bytes_Lower_Band'] = data['Bytes_30D_MA'] - (2 * data['Bytes_30D_STD'])

plt.figure(figsize=(12, 6))
plt.plot(data['date'], data['Bytes'], label='Daily Bytes', color='red', alpha=0.6)
plt.plot(data['date'], data['Bytes_30D_MA'], label='30-Day Moving Average (Bytes)', color='blue', linewidth=2)
plt.fill_between(data['date'], data['Bytes_Upper_Band'], data['Bytes_Lower_Band'], 
                 color='grey', alpha=0.3, label='Bollinger Bands')
plt.yscale('log')
plt.title('Daily Bytes, 30-Day Moving Average, and Bollinger Bands (Log Scale)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Bytes (Log Scale)', fontsize=12)

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=10, frameon=False)

plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()